# Delay model: data, evaluation, new ideas

What the app predicts, how we know whether it is any good, and a place to try something new without breaking anything.

**The question.** A train has just left a stop (a *real* departure, with its real delay). How late will it be at each of the next stops? The app answers with one of three models:

| model | what it does |
| --- | --- |
| `raw` | the announced delay, held on every stop |
| `rules` | the delay trains typically recover on the way (constants read from `js/modules/model.js`) |
| `tables` | the rules plus the usual error of each pair of stations, learned from past days |

**How to read the scores.** The unit is the mean absolute error, in minutes, over the next 1 to 6 stops, against the **real** arrival times. Every score is computed on a day the model has not seen: the models learn from other days, then are tested on one more.

The notebook uses `scripts/evaluate.py` for everything the app also runs, so it can never drift from it. Run the cells in order; only the first two are meant to be edited.

## 1. Setup

In [ ]:
import json
import os
import statistics
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# the repository root, found from wherever the notebook is opened
REPO = Path.cwd()
while not (REPO / 'scripts' / 'evaluate.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'scripts'))
import evaluate as E  # the same code the app's scores come from

# >>> the export to study (from the collector). TRENO_EXPORT overrides it.
EXPORT = Path(os.environ.get('TRENO_EXPORT', Path.home() / 'Downloads' / 'export.json'))

# the constants of the rules, read from the app itself
CONSTANTS = E.recovery_constants(E.MODEL_JS.read_text())
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)
NAMES = {code: v[2] for code, v in json.loads((REPO / 'data' / 'stations.json').read_text()).items()}  # station code -> name
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})
print('repository:', REPO)
print('export    :', EXPORT)
print('rules     :', CONSTANTS)

## 2. The data

One row per train, each with its stops and, for each stop, the scheduled and the real times when ViaggiaTreno gave them. Real times are not always there: arrivals especially (the table below shows how often).

In [ ]:
export = json.loads(EXPORT.read_text())
trains = export['trains']
summary = E.data_summary(trains)
print(f"{summary['trains']} trains, {summary['stops']} stops, {summary['polls_per_train']} polls per train")
print(f"real arrivals {summary['real_arrival_share']:.0%}, real departures {summary['real_departure_share']:.0%}")

rows = E.build_pairs(trains)  # (real departure at a stop, real arrival at a later stop)
df = pd.DataFrame(rows)
whole = E.usable_days(rows)
per_day = df.groupby('day')['train'].nunique().rename('trains').to_frame()
per_day['pairs'] = df.groupby('day').size()
per_day['whole day'] = per_day.index.isin(whole)
per_day

A *whole day* has enough trains and at least half of a usual day: an export made in the morning gives a fragment of the last day, which is not a fair test (early trains behave differently), so it is kept out of the scores.

In [ ]:
ax = per_day['trains'].plot.bar(color=['#4fd1a1' if w else '#9fb0c7' for w in per_day['whole day']], title='Trains per day (green: whole day, used for scoring)')
ax.set_ylabel('trains')
plt.xticks(rotation=0)
plt.show()

## 3. The problem, in one picture

For every pair, `r0` is the delay when the train left the stop we know, and `y` the delay it really had at a stop `k` stops later. If trains simply kept their delay (`raw`), all points would sit on the diagonal. They sit below it: **trains recover time**, and the rules model that.

In [ ]:
sample = df[(df['k'] <= 6) & (df['r0'] <= 25) & (df['y'].between(-5, 30))]
fig, ax = plt.subplots()
hb = ax.hexbin(sample['r0'], sample['y'], gridsize=40, bins='log', cmap='viridis', mincnt=1)
x = np.linspace(0, 25, 100)
ax.plot(x, x, 'w--', label='raw: the delay is kept')
rules_curve = [E.predict_rules({'r0': v, 'h': 15}, CONSTANTS) for v in x]
ax.plot(x, rules_curve, color='#f5c542', lw=2, label='rules (15 min ahead)')
ax.set(xlabel='delay when the train left the known stop (min)', ylabel='delay at a later stop (min)', title='Delay now vs delay later')
ax.legend()
fig.colorbar(hb, label='pairs (log)')
plt.show()

## 4. The three models, on a day they have not seen

The last whole day is the test. The rules have their constants from `model.js`; the tables are built **only from the days before** the test day.

In [ ]:
calibration_days, test_day = E.split_days(rows)
calibration = [r for r in rows if r['day'] in calibration_days]
test = [r for r in rows if r['day'] == test_day]
table = E.build_corrections(calibration, CONSTANTS)
predictors = {'raw': E.predict_raw, 'rules': E.predict_rules, 'tables': E.tables_predictor(table)}
print(f'learned from {calibration_days}  ->  tested on {test_day}  ({len(test)} pairs)')
print(f"the table: {len(table['pairs'])} pairs of stations, {len(table['stations'])} stations")

scores = {name: E.score(test, predict, CONSTANTS) for name, predict in predictors.items()}
horizons = pd.DataFrame({name: s['mae_by_horizon'] for name, s in scores.items()})
horizons.index.name = 'stops ahead'
summary_table = pd.DataFrame({name: {k: s[k] for k in ('mae_1_to_6', 'median_abs', 'p90_abs', 'bias')} for name, s in scores.items()}).T
summary_table.columns = ['mean error', 'median error', '90th percentile', 'bias']
display(summary_table) if 'display' in globals() else print(summary_table)
horizons.plot.bar(rot=0, title='Mean error by number of stops ahead (min)', color=['#9fb0c7', '#f5c542', '#4fd1a1'])
plt.ylabel('minutes')
plt.show()

**Bias** is the mean of *prediction minus truth*: a positive value means the model is late on average. The models aim at the median of the error, which is what minimises the mean absolute error; a skewed distribution (a few very late trains) leaves a small bias on purpose.

## 5. Is it the same every day?

Each whole day is scored in turn with tables built from the **other** days. One good day proves little; a gain on every day is evidence.

In [ ]:
days = [d for d in whole]
daily = pd.DataFrame(E.per_day_scores(rows, days, CONSTANTS)).set_index('day')
daily['tables vs rules'] = (1 - daily['tables'] / daily['rules-v2']).map('{:.1%}'.format)
daily['tables vs raw'] = (1 - daily['tables'] / daily['raw']).map('{:.1%}'.format)
display(daily) if 'display' in globals() else print(daily)
daily[['raw', 'rules-v2', 'tables']].plot.bar(rot=0, title='Mean error per day, each day scored with what it had not seen', color=['#9fb0c7', '#f5c542', '#4fd1a1'])
plt.ylabel('minutes')
plt.show()

## 6. How far off can an estimate be?

A single number is not enough: the app also shows a range (the 5th to the 95th percentile of the error, so 9 times in 10 the truth is inside). A range is only worth showing if it holds on a new day, which is the `coverage` column.

In [ ]:
intervals = E.tables_intervals(calibration, test, CONSTANTS, calibration_days)
iv = pd.DataFrame(intervals).T
iv['width'] = iv['hi'] - iv['lo']
iv.index.name = 'stops ahead'
display(iv) if 'display' in globals() else print(iv)
fig, ax = plt.subplots()
ks = [int(k) for k in iv.index]
ax.fill_between(ks, iv['lo'], iv['hi'], color='#4fd1a1', alpha=0.3, label='90 % range around the estimate')
ax.axhline(0, color='white', lw=1)
ax.set(xlabel='stops ahead', ylabel='real minus predicted (min)', title='Range of the error (tables)')
ax.legend()
plt.show()

## 7. Where do the errors come from?

The residual is what is left: real delay minus the tables' prediction. If it still depends on something we know (a station, the hour, the type of train), a better model exists.

In [ ]:
predict = predictors['tables']
res = pd.DataFrame(test)
res['pred'] = [predict(r, CONSTANTS) for r in test]
res['residual'] = res['y'] - res['pred']
res = res[res['k'] <= 6]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
res['residual'].clip(-12, 15).plot.hist(bins=60, ax=axes[0], color='#4fd1a1')
axes[0].set(title='Residual of the tables (min)', xlabel='real - predicted')
res.groupby('hour')['residual'].agg(['median']).plot(ax=axes[1], legend=False, marker='o', color='#f5c542')
axes[1].set(title='Median residual by hour of departure', xlabel='hour', ylabel='min')
res.groupby('k')['residual'].median().plot(ax=axes[2], marker='o', color='#9fb0c7')
axes[2].set(title='Median residual by stops ahead', xlabel='stops ahead', ylabel='min')
plt.tight_layout()
plt.show()

stations = res.groupby('to')['residual'].agg(['median', 'count'])
stations = stations[stations['count'] >= 15].sort_values('median')
stations.insert(0, 'station', [NAMES.get(code, code).title() for code in stations.index])
print('stops where the tables are still most wrong (median residual, min):')
display(pd.concat([stations.head(5), stations.tail(5)])) if 'display' in globals() else print(pd.concat([stations.head(5), stations.tail(5)]))

## 8. Why the tables work: stretches have habits

For each pair of stations (last known stop, stop ahead) the rules are off by about the same amount every day: a stretch whose timetable has slack, one that is always slow. If the habit were noise, the pairs would not agree from one day to the next. They do:

In [ ]:
by_pair = {}
for d in days:
    day_rows = [r for r in rows if r['day'] == d and r['k'] <= 6]
    cell = {}
    for r in day_rows:
        cell.setdefault((r['from'], r['to']), []).append(r['y'] - E.predict_rules(r, CONSTANTS))
    by_pair[d] = {k: statistics.median(v) for k, v in cell.items() if len(v) >= 3}

if len(days) >= 2:
    a, b = days[-2], days[-1]
    shared = sorted(set(by_pair[a]) & set(by_pair[b]))
    if len(shared) > 5:
        xa = np.array([by_pair[a][k] for k in shared])
        xb = np.array([by_pair[b][k] for k in shared])
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.scatter(xa, xb, s=8, alpha=0.4, color='#4fd1a1')
        lim = max(abs(xa).max(), abs(xb).max(), 1)
        ax.plot([-lim, lim], [-lim, lim], 'w--')
        ax.set(xlabel=f'median error of the rules, {a}', ylabel=f'median error of the rules, {b}', title=f'{len(shared)} pairs of stations seen on both days')
        plt.show()
        print(f'correlation between the two days: {np.corrcoef(xa, xb)[0, 1]:.2f}')
    else:
        print('not enough pairs seen on both days to compare')
else:
    print('needs two whole days')

## 9. A harness to try new models

`leave_one_day_out` scores any model on every whole day with a version of it that has not seen that day. A model is a function `fit(train_rows)` that returns `predict(rows) -> array of delays (min)`.

Rows have: `day`, `train`, `k` (stops ahead), `r0` (delay when the known stop was left), `h` (scheduled minutes between the known stop and the stop ahead), `y` (the truth), `from`, `to` (station codes), `hour` (of departure), `cat` (category of the train).

In [ ]:
def leave_one_day_out(fit, days=days, rows=rows):
    """Mean error over 1 to 6 stops ahead, for each whole day, with a model that did not see that day."""
    out = {}
    for d in days:
        train_rows = [r for r in rows if r['day'] in days and r['day'] != d]
        test_rows = [r for r in rows if r['day'] == d and r['k'] <= 6]
        if not train_rows or not test_rows:
            continue
        predict = fit(train_rows)
        out[d] = float(np.mean(np.abs(np.asarray(predict(test_rows)) - np.array([r['y'] for r in test_rows]))))
    return out


def per_row(f):
    return lambda rows_: np.array([f(r) for r in rows_])


def fit_raw(train_rows):
    return per_row(lambda r: E.predict_raw(r, CONSTANTS))


def fit_rules(train_rows):
    return per_row(lambda r: E.predict_rules(r, CONSTANTS))


def fit_tables(train_rows):
    predict = E.tables_predictor(E.build_corrections(train_rows, CONSTANTS))
    return per_row(lambda r: predict(r, CONSTANTS))


candidates = {'raw': fit_raw, 'rules': fit_rules, 'tables': fit_tables}
results = pd.DataFrame({name: leave_one_day_out(fit) for name, fit in candidates.items()})
results.loc['mean'] = results.mean()
display(results.round(3)) if 'display' in globals() else print(results.round(3))

### An idea to test: does anything but the stations help?

Gradient boosting on what the app knows about a pair that is not a station: the delay, the time to the stop, the horizon, the hour, the type of train. It learns the error of the **rules**. If it does not beat the tables, the stations carry the signal; when a genuinely new feature is available (the delay of the train just ahead on the line, say), add it to `features` and run the same cell.

In [ ]:
try:
    from sklearn.ensemble import HistGradientBoostingRegressor
except ImportError:
    HistGradientBoostingRegressor = None
    print('scikit-learn is not installed: pip install scikit-learn')


def features(r, cats):
    return [r['r0'], r['h'], r['k'], r['hour'], cats.get(r['cat'], -1)]


def fit_boosting(train_rows):
    cats = {c: i for i, c in enumerate(sorted({r['cat'] for r in train_rows}))}
    X = np.array([features(r, cats) for r in train_rows])
    target = np.array([r['y'] - E.predict_rules(r, CONSTANTS) for r in train_rows])  # the error of the rules
    model = HistGradientBoostingRegressor(loss='absolute_error', max_iter=120, learning_rate=0.08, max_leaf_nodes=16, min_samples_leaf=60, random_state=0).fit(X, target)

    def predict(rows_):
        base = np.array([E.predict_rules(r, CONSTANTS) for r in rows_])
        return np.maximum(E.TABLES_FLOOR, base + model.predict(np.array([features(r, cats) for r in rows_])))
    return predict


if HistGradientBoostingRegressor is not None:
    candidates['boosting (no stations)'] = fit_boosting
    results = pd.DataFrame({name: leave_one_day_out(fit) for name, fit in candidates.items()})
    results.loc['mean'] = results.mean()
    display(results.round(3)) if 'display' in globals() else print(results.round(3))

## 10. Do the rules need a new calibration?

The three constants of the rules (and the offset) were fitted once. With more days, are they still right? Each whole day is scored with constants refitted on the other days, against the constants in the app.

In [ ]:
refit = {}
for d in days:
    others = [r for r in rows if r['day'] in days and r['day'] != d]
    if not others:
        continue
    fitted = E.fit_constants(others, CONSTANTS)
    day_rows = [r for r in rows if r['day'] == d and r['k'] <= 6]
    refit[d] = {
        'app constants': np.mean([abs(E.predict_rules(r, CONSTANTS) - r['y']) for r in day_rows]),
        'refitted on the other days': np.mean([abs(E.predict_rules(r, fitted) - r['y']) for r in day_rows]),
        **{f'fitted {k}': v for k, v in fitted.items()},
    }
refit = pd.DataFrame(refit).T
display(refit.round(3)) if 'display' in globals() else print(refit.round(3))
print('If the first two columns are the same, there is nothing to gain: leave the constants alone.')

## 11. Shipping a model

Nothing in this notebook changes the app. To publish new scores and a new table:

```bash
python3 scripts/evaluate.py path/to/export.json
```

It writes `data/model-report.json` (what the Lab page shows) and `data/segment-corrections.json` (the table behind `tables`). A new model idea that wins here on every day still has to be written in `js/modules/model.js`, registered in `js/modules/models.js`, and scored by `scripts/evaluate.py`: the app only recommends a model that beats the baseline on a day it has not seen.

**Ideas that need more data than the collector gives today**

- the delay of the train just ahead on the same line (a late train holds the next one back);
- the position between two stations, to correct a prediction with how far the train really got;
- weekends, holidays and strikes, once a few weeks are collected.